# 03 — Features & temporal split (leakage-safe)

**Input:** `nodes_clean.csv` (from `02`) and `edges.csv`.
**Output:** `features.csv` — one row per track with a `split` label, three horizon
outcomes, and 51 features, every one computable **from information available at the
track's own posting instant**.

This is the notebook where the thesis can quietly go wrong, so every guard is explicit.

**Three rules enforced throughout**

1. *As-of-post-time.* A feature for a track posted at time `T` may use only events
   strictly before `T`. Remixes are dated by the **child's** post time, so a remix
   only "exists" for feature purposes once its child was posted.
2. *Fit vocabularies on train only.* The tag list and licence categories are learned
   from training rows, then applied to test — the test distribution never leaks back.
3. *No snapshot columns as features.* `is_remixed`, `n_remixes`, `out_degree_*`,
   `first_remix_t` are outcomes/"today" values and are never fed to the model.

**Key parameters** (change these two lines to re-run any scenario):
`H_PRIMARY = 365` days, `CUTOFF = 2022-01-01`. Alternates `180` / `730` are built
as extra label columns so the horizon is a reporting choice, not a rebuild.

In [1]:
import numpy as np, pandas as pd
from bisect import bisect_left, insort
from collections import Counter

NODES="data/processed/nodes_clean.csv"; EDGES="data/processed/edges.csv"; OUT="data/processed/features.csv"
DAY=86400
H_PRIMARY=365; CUTOFF="2022-01-01"        # <-- the two decisions
HORIZONS=[180,365,730]

nodes=pd.read_csv(NODES); edges=pd.read_csv(EDGES)
nodes["t"]=nodes.date_unix.astype("int64")          # post time (unix s), already correct
CRAWL_END=int(nodes.t.max())
nodes["obs_days"]=(CRAWL_END-nodes.t)//DAY

loc=edges[edges.edge_type=="local"].copy()
first_remix=loc.groupby("parent_id").child_date_unix.min()
nodes["first_remix_t"]=nodes.upload_id.map(first_remix)     # NaN = never remixed
print("loaded", nodes.shape, "| crawl end", pd.to_datetime(CRAWL_END,unit='s').date())

loaded (51486, 27) | crawl end 2026-07-20


## 1. Horizon labels + observation filter

`y_H = 1` iff the first local remix arrived within `H` days of posting. A track is
only *fairly labelled* for horizon `H` if it has had at least `H` days of observation
before the crawl (`valid_H`) — otherwise a `0` might just mean "not observed long
enough". The labels are nested (`y_180 ⊆ y_365 ⊆ y_730`) by construction.

In [2]:
lat=(nodes.first_remix_t-nodes.t)/DAY
for H in HORIZONS:
    nodes[f"y_{H}"]=(lat<=H).fillna(False)
    nodes[f"valid_{H}"]=nodes.obs_days>=H
v=nodes[f"valid_{max(HORIZONS)}"]
assert (nodes.loc[v,"y_180"]<=nodes.loc[v,"y_365"]).all()
assert (nodes.loc[v,"y_365"]<=nodes.loc[v,"y_730"]).all()
for H in HORIZONS:
    m=nodes[f"valid_{H}"]; print(f"y_{H}: base={100*nodes.loc[m,f'y_{H}'].mean():.1f}%  valid_N={int(m.sum())}")

y_180: base=25.9%  valid_N=51134
y_365: base=28.4%  valid_N=50728
y_730: base=30.8%  valid_N=49742


## 2. Temporal split with an embargo

- **`train`:** tracks whose full primary-horizon outcome window closed **before** the cutoff (`t + H_PRIMARY < CUTOFF`). That is 46,065 tracks posted from 2004-10-28 to 2020-12-31, with a base rate of 27.8 %.
- **`test`:** tracks posted **on or after** the cutoff that still have a full primary window before the crawl. That is 3,330 tracks posted from 2022-01-01 to 2025-07-20, with a base rate of 33.1 %.
- **Embargo:** the 366-day gap between them stops any training track's 365-day outcome window from reaching into the test period.
- **Dropped:** tracks in the gap, or too recent to be fairly labelled (2,091).

**Other horizons.** The split column is defined for the primary 365-day horizon. For the 730-day horizon, `04` shortens the training set further (`train_rows(H)`) so that every training label window still closes before 2022. The 180- and 365-day training sets are identical.

In [3]:
C=int(pd.Timestamp(CUTOFF,tz='UTC').timestamp()); Hp=H_PRIMARY*DAY
test_end=CRAWL_END-Hp
split=np.full(len(nodes),"drop",object)
split[(nodes.t< C-Hp).values]="train"
split[((nodes.t>=C)&(nodes.t<=test_end)).values]="test"
nodes["split"]=split
nodes.loc[~nodes[f"valid_{H_PRIMARY}"],"split"]="drop"

print(nodes.split.value_counts().to_string())
for s in ["train","test"]:
    m=nodes.split==s; d=pd.to_datetime(nodes.loc[m,'t'],unit='s')
    print(f"  {s}: n={int(m.sum())}  y_{H_PRIMARY} base={100*nodes.loc[m,f'y_{H_PRIMARY}'].mean():.1f}%"
          f"  {d.min().date()}..{d.max().date()}")
gap=(nodes.loc[nodes.split=='test','t'].min()-nodes.loc[nodes.split=='train','t'].max())/DAY
assert gap>=H_PRIMARY-1, "embargo violated"; print(f"embargo gap: {gap:.0f}d (>= {H_PRIMARY})")

split
train    46065
test      3330
drop      2091
  train: n=46065  y_365 base=27.8%  2004-10-28..2020-12-31
  test: n=3330  y_365 base=33.1%  2022-01-01..2025-07-20
embargo gap: 366d (>= 365)


## 3. Author-history features (as-of `T`)

All computed from the author's activity *before* `T`. Counts of remixes received /
given are as-of: a remix counts only once its child was posted (`child_date < T`).

In [4]:
id2user=nodes.set_index("upload_id").user_name
nodes=nodes.sort_values(["user_name","t"]).reset_index(drop=True)
g=nodes.groupby("user_name",sort=False)
nodes["f_author_prior_uploads"]=g.cumcount()
nodes["_first_t"]=g["t"].transform("min")
nodes["f_author_tenure_days"]=(nodes.t-nodes._first_t)/DAY
nodes["f_author_days_since_last"]=((nodes.t-g["t"].shift(1))/DAY)
nodes["f_author_is_new"]=(nodes.f_author_prior_uploads==0).astype(int)

loc2=loc[loc.parent_id.isin(id2user.index)&loc.child_id.isin(id2user.index)]
recv=pd.DataFrame({"a":loc2.parent_id.map(id2user).values,"et":loc2.child_date_unix.values})
give=pd.DataFrame({"a":loc2.child_id.map(id2user).values ,"et":loc2.child_date_unix.values})
def asof(ev,authors,times):
    d={a:np.sort(s.et.values) for a,s in ev.groupby("a")}; out=np.zeros(len(authors),int)
    for i,(a,t) in enumerate(zip(authors,times)):
        arr=d.get(a); out[i]=0 if arr is None else bisect_left(arr,t)
    return out
nodes["f_author_prior_remixes_received"]=asof(recv,nodes.user_name.values,nodes.t.values)
nodes["f_author_prior_remixes_given"]   =asof(give,nodes.user_name.values,nodes.t.values)

frt=nodes.first_remix_t.values.astype("float64")
succ=np.zeros(len(nodes)); prem=np.zeros(len(nodes))
for a,idx in nodes.groupby("user_name",sort=False).indices.items():
    seen=[]
    for j in list(idx):
        t=nodes.t.values[j]; c=bisect_left(seen,t); prem[j]=c
        pu=nodes.f_author_prior_uploads.values[j]; succ[j]=c/pu if pu>0 else 0.0
        if not np.isnan(frt[j]): insort(seen,frt[j])
nodes["f_author_prior_remixed_track_count"]=prem.astype(int)
nodes["f_author_prior_success_rate"]=succ
print("author-history features done")

author-history features done


## 4. Track-intrinsic + source-prominence + tag/licence features

Intrinsic attributes are fixed at upload. Source ("parent") prominence is again
as-of: how remixed were this track's own sources *at the moment it was posted*.
Tag vocabulary (top-20) and licence categories are fit on **train only**.

In [5]:
idset=set(nodes.upload_id)
nodes["f_n_sources"]=nodes.n_sources; nodes["f_in_degree_local"]=nodes.in_degree_local
nodes["f_is_derivative"]=(nodes.n_sources>0).astype(int); nodes["f_num_files"]=nodes.num_files
nodes["f_title_len"]=nodes.name.fillna("").astype(str).str.len()
nodes["f_title_words"]=nodes.name.fillna("").astype(str).str.split().str.len()
def toks(s):
    return [] if pd.isna(s) else [x.strip() for x in str(s).replace(";",",").split(",") if x.strip()]
nodes["_ut"]=nodes.usertags.apply(toks); nodes["_st"]=nodes.systags.apply(toks)
nodes["f_n_usertags"]=nodes._ut.str.len(); nodes["f_has_usertags"]=(nodes.f_n_usertags>0).astype(int)
nodes["f_sys_flac"]=nodes._st.apply(lambda L:int("flac" in L))
dtp=pd.to_datetime(nodes.t,unit="s",utc=True)
nodes["f_post_year"]=dtp.dt.year; nodes["f_post_month"]=dtp.dt.month; nodes["f_post_dow"]=dtp.dt.dayofweek

par_children={p:np.sort(s.child_date_unix.values) for p,s in loc.groupby("parent_id")}
child2parents=loc2.groupby("child_id").parent_id.apply(list).to_dict()
pmax=np.zeros(len(nodes)); pmean=np.zeros(len(nodes)); uid=nodes.upload_id.values; tt=nodes.t.values
for i in range(len(nodes)):
    ps=child2parents.get(uid[i])
    if not ps: continue
    vals=[bisect_left(par_children.get(p,np.empty(0)),tt[i]) for p in ps]
    pmax[i]=max(vals); pmean[i]=float(np.mean(vals))
nodes["f_parent_max_prior_remixes"]=pmax.astype(int); nodes["f_parent_mean_prior_remixes"]=pmean

tr=nodes.split=="train"
cnt=Counter();  [cnt.update(set(L)) for L in nodes.loc[tr,"_ut"]]
TOPK=[t for t,_ in cnt.most_common(20)]
for tg in TOPK: nodes[f"f_tag_{tg}"]=nodes._ut.apply(lambda L,tg=tg:int(tg in L))
lic_top=nodes.loc[tr,"license"].value_counts().head(8).index.tolist()
nodes["_lic"]=np.where(nodes.license.isin(lic_top),nodes.license,"other")
nodes=pd.concat([nodes,pd.get_dummies(nodes["_lic"],prefix="f_lic").astype(int)],axis=1)
nodes["f_author_days_since_last"]=nodes["f_author_days_since_last"].fillna(-1)
feat_cols=[c for c in nodes.columns if c.startswith("f_")]
print(len(feat_cols),"features |","top tags:",TOPK[:6],"...")

51 features | top tags: ['male_vocals', 'guitar', 'drums', 'female_vocals', 'electronic', 'bass'] ...


## 5. Leakage self-checks

If any of these fail, a feature is peeking at the future — stop and fix before modelling.

In [6]:
banned=["is_remixed","n_remixes","out_degree","first_remix","y_180","y_365","y_730"]
assert not any(any(b in c for b in banned) for c in feat_cols), "outcome leaked into features"
first=nodes[nodes.f_author_prior_uploads==0]
assert (first.f_author_prior_remixes_received==0).all() and (first.f_author_tenure_days==0).all()
assert (first.f_author_prior_success_rate==0).all()
assert nodes.f_author_prior_success_rate.between(0,1).all()
a=recv.a.value_counts().index[0]; sub=nodes[nodes.user_name==a].sort_values("t")
assert sub.f_author_prior_remixes_received.is_monotonic_increasing
assert nodes[feat_cols].isna().sum().sum()==0
print("all leakage self-checks passed  |  earliest tracks carry zero history, counts monotone, no NaNs")

all leakage self-checks passed  |  earliest tracks carry zero history, counts monotone, no NaNs


## 6. Assemble `features.csv` and run a leakage smell-test

A temporally split model that scores *too* well is the classic sign of leakage. A believable AUC, and no single near-perfect feature, are the evidence that the pipeline is honest.

**Result.** A default gradient-boosting model reaches AUC 0.840, with an AP of 0.676 against a base rate of 0.331. The strongest single feature reaches AUC 0.692. Both are believable, not close to 1.0. The tuned model in `04` reaches 0.842.

In [7]:
keep=["upload_id","user_name","split","obs_days"]+[f"y_{H}" for H in HORIZONS]+\
     [f"valid_{H}" for H in HORIZONS]+feat_cols
out=nodes[keep].copy(); out.to_csv(OUT,index=False)
print("wrote",OUT,out.shape)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
tr_=out[out.split=="train"]; te_=out[out.split=="test"]
Xtr,Xte=tr_[feat_cols].values,te_[feat_cols].values
ytr,yte=tr_[f"y_{H_PRIMARY}"].astype(int),te_[f"y_{H_PRIMARY}"].astype(int)
m=HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,
   l2_regularization=1.0,random_state=0).fit(Xtr,ytr)
p=m.predict_proba(Xte)[:,1]
print(f"HGB temporal test  AUC={roc_auc_score(yte,p):.3f}  AP={average_precision_score(yte,p):.3f}"
      f"  (base {yte.mean():.3f})  -- believable, not ~1.0")
aucs={c:max(a,1-a) for c in feat_cols for a in [roc_auc_score(ytr,tr_[c].astype(float))] if tr_[c].nunique()>1}
print("strongest single feature AUC:",round(max(aucs.values()),3),"(none should be ~1.0)")
print("top signals:",", ".join(c for c,_ in sorted(aucs.items(),key=lambda x:-x[1])[:5]))

wrote data/processed/features.csv (51486, 61)
HGB temporal test  AUC=0.840  AP=0.676  (base 0.331)  -- believable, not ~1.0
strongest single feature AUC: 0.692 (none should be ~1.0)
top signals: f_is_derivative, f_n_sources, f_author_prior_success_rate, f_in_degree_local, f_parent_mean_prior_remixes


## 7. Notes for `04` (modelling)

- **Horizons:** the feature matrix carries **all three horizons**. Select `y_180` / `y_365` / `y_730` and mask on the matching `valid_H`. The training set for each horizon is embargoed in `04`.
- **Covariate shift:** expect a real shift between training and test, because test-era authors are far more prolific (49 accounts produce 63 % of test tracks). This is genuine drift over time, not a bug, and a temporally split evaluation is meant to test generalisation across it.
- **Correlated features:** `f_n_sources`, `f_in_degree_local` and `f_is_derivative` describe the same property. Single-feature importance is diluted across them, so `04` also reports grouped importance.
- **Changing the setup:** to try another split, change `CUTOFF`; to change the question, change `H_PRIMARY`.
- **Forbidden inputs:** never add a feature that reads `is_remixed`, `n_remixes`, `num_scores`, or any `out_degree` / `first_remix` column. These are outcomes or current snapshot values.